In [ ]:
"""
=============================================================================
 Absorber Column Validation
=============================================================================

"""

import numpy as np
import warnings
from absorber import Absorber

warnings.filterwarnings("ignore", category=RuntimeWarning)
import io, sys

def suppress_print(func, *args, **kwargs):
    old = sys.stdout
    sys.stdout = io.StringIO()
    try:
        return func(*args, **kwargs)
    finally:
        sys.stdout = old


print("=" * 70)
print("  ABSORBER MODEL VALIDATION")
print("=" * 70)

# =====================================================================
#  TEST 1: Mass Balance Closure (CO₂ and H₂O)
# =====================================================================
print("\n  ─── TEST 1: Mass Balance Closure ───")

col = Absorber(c_NaOH_in=2.0, L_vol_flow=40.0, Z=15.0, disc_z=401)
suppress_print(col.solve_column)

# CO₂ balance
n_CO2_gas = col.G_inert * col.A_col * (col.Y_CO2_in - col.Y_CO2[-1])
u_L_out = col._local_u_L(col.C_DIC[0])
n_CO2_liq = (
    u_L_out * col.C_DIC[0] * 1000.0 * col.A_col
    - col.u_L * col.C_DIC_top * 1000.0 * col.A_col
)
err_CO2 = abs(n_CO2_gas - n_CO2_liq) / max(abs(n_CO2_gas), 1e-20)

# H₂O balance
from scipy.integrate import simpson
n_H2O_gas = col.G_inert * col.A_col * (col.Y_H2O[-1] - col.Y_H2O[0])
n_H2O_int = simpson(col.a_w * col.N_H2O, x=col.l_eval) * col.A_col
err_H2O = abs(n_H2O_gas - n_H2O_int) / max(abs(n_H2O_gas), 1e-20)

print(f"    CO₂ removed (gas-phase):  {n_CO2_gas:.6e} mol/s")
print(f"    CO₂ gained  (liquid):     {n_CO2_liq:.6e} mol/s")
print(f"    CO₂ balance error:        {err_CO2:.4e}  "
      f"{'✓' if err_CO2 < 1e-3 else '✗'}")
print(f"    H₂O gained  (gas-phase):  {n_H2O_gas:.6e} mol/s")
print(f"    H₂O ∫N_H2O dz:           {n_H2O_int:.6e} mol/s")
print(f"    H₂O balance error:        {err_H2O:.4e}  "
      f"{'✓' if err_H2O < 0.01 else '✗'}")


# =====================================================================
#  TEST 2: Limiting Cases
# =====================================================================
print("\n  ─── TEST 2: Limiting Cases ───")

# Case A: Very short column (Z → 0) → minimal capture
col_short = Absorber(c_NaOH_in=2.0, L_vol_flow=40.0, Z=0.1, disc_z=51)
suppress_print(col_short.solve_column)
eta_short = col_short.capture_efficiency()
print(f"    Z = 0.1 m:  η = {eta_short:.2f}%  "
      f"{'✓' if eta_short < 30 else '✗'} (should be low)")

# Case B: Very tall column → near-complete capture
col_tall = Absorber(c_NaOH_in=2.0, L_vol_flow=40.0, Z=30.0, disc_z=401)
suppress_print(col_tall.solve_column)
eta_tall = col_tall.capture_efficiency()
print(f"    Z = 30  m:  η = {eta_tall:.2f}%  "
      f"{'✓' if eta_tall > 99 else '✗'} (should be ~100%)")

# Case C: Very dilute NaOH → lower capture than concentrated
col_dilute = Absorber(c_NaOH_in=0.01, L_vol_flow=40.0, Z=15.0, disc_z=201)
suppress_print(col_dilute.solve_column)
eta_dilute = col_dilute.capture_efficiency()

col_conc = Absorber(c_NaOH_in=2.0, L_vol_flow=40.0, Z=15.0, disc_z=201)
suppress_print(col_conc.solve_column)
eta_conc = col_conc.capture_efficiency()
print(f"    c_NaOH=0.01 M:  η = {eta_dilute:.2f}%")
print(f"    c_NaOH=2.00 M:  η = {eta_conc:.2f}%")
print(f"    Dilute < Concentrated:  "
      f"{'✓' if eta_dilute < eta_conc else '✗'}")

# =====================================================================
#  TEST 3: Monotonicity of Profiles
# =====================================================================
print("\n  ─── TEST 4: Profile Monotonicity ───")

# y_CO2 should decrease from bottom (z=0) to top (z=Z)
y_decreasing = np.all(np.diff(col_ref.y_CO2) <= 1e-15)
print(f"    y_CO₂ monotonically decreasing:  "
      f"{'✓' if y_decreasing else '✗'}")

# C_DIC should decrease from bottom (z=0) to top (z=Z)
dic_decreasing = np.all(np.diff(col_ref.C_DIC) <= 1e-15)
print(f"    C_DIC monotonically decreasing:  "
      f"{'✓' if dic_decreasing else '✗'}")

# pH should increase from bottom to top (less DIC → more alkaline)
ph_increasing = np.all(np.diff(col_ref.pH) >= -1e-10)
print(f"    pH monotonically increasing:     "
      f"{'✓' if ph_increasing else '✗'}")


# =====================================================================
#  TEST 4: Analytical NTU Cross-Check
# =====================================================================
print("\n  ─── TEST 5: NTU / HTU Cross-Check ───")

# For a very reactive system (Ha >> 1), KG ≈ kG
# NTU_approx = ln(y_in / y_out) for y* ≈ 0
y_in  = col_ref.y_CO2_in
y_out = col_ref.y_CO2[-1]
if y_out > 1e-12:
    NTU_approx = np.log(y_in / y_out)
else:
    NTU_approx = np.nan

print(f"    NTU (model):        {col_ref.NTU:.3f}")
print(f"    NTU (approx ln):    {NTU_approx:.3f}")
if not np.isnan(NTU_approx) and not np.isnan(col_ref.NTU):
    ntu_ratio = col_ref.NTU / NTU_approx
    print(f"    Ratio model/approx: {ntu_ratio:.3f}  "
          f"{'✓' if 0.5 < ntu_ratio < 2.0 else '⚠ check'}")

print(f"    HTU (model):        {col_ref.HTU:.3f} m")
if not np.isnan(col_ref.HTU):
    Z_check = col_ref.NTU * col_ref.HTU
    print(f"    NTU × HTU:          {Z_check:.3f} m  "
          f"(Z = {col_ref.Z:.1f} m)  "
          f"{'✓' if abs(Z_check - col_ref.Z) / col_ref.Z < 0.01 else '✗'}")


# =====================================================================
#  TEST 5: BVP Convergence Check
# =====================================================================
print("\n  ─── TEST 6: BVP Convergence ───")
print(f"    bvp_converged flag:        {col_ref.bvp_converged}  "
      f"{'✓' if col_ref.bvp_converged else '✗'}")
C_DIC_top_actual = col_ref.C_DIC[-1]
print(f"    C_DIC(z=Z) actual:         {C_DIC_top_actual:.2e} mol/L")
print(f"    C_DIC(z=Z) target:         {col_ref.C_DIC_top:.2e} mol/L")
bc_err = abs(C_DIC_top_actual - col_ref.C_DIC_top)
print(f"    BC residual:               {bc_err:.2e}  "
      f"{'✓' if bc_err < 1e-6 else '✗'}")


# =====================================================================
#  TEST 6: Sensitivity Consistency (η should not exceed 100%)
# =====================================================================
print("\n  ─── TEST 7: Physical Bounds ───")

eta = col_ref.capture_efficiency()
print(f"    0 ≤ η ≤ 100:  η = {eta:.4f}%  "
      f"{'✓' if 0 <= eta <= 100 else '✗'}")

util = col_ref.NaOH_utilization()
print(f"    0 ≤ util ≤ 1:  util = {util:.6f}  "
      f"{'✓' if 0 <= util <= 1 else '✗'}")

all_positive_pH = np.all(col_ref.pH > 0) and np.all(col_ref.pH < 15)
print(f"    0 < pH < 15:   "
      f"{'✓' if all_positive_pH else '✗'}  "
      f"(range: {col_ref.pH.min():.2f} – {col_ref.pH.max():.2f})")

all_positive_E = np.all(col_ref.E >= 1.0)
print(f"    E ≥ 1:         "
      f"{'✓' if all_positive_E else '✗'}  "
      f"(range: {col_ref.E.min():.2f} – {col_ref.E.max():.2f})")


print("\n" + "=" * 70)
print("  VALIDATION COMPLETE")
print("=" * 70)

  ABSORBER MODEL VALIDATION

  ─── TEST 1: Mass Balance Closure ───
  Auto-size (gas flooding): D = 29 cm, u_G = 14.7 cm/s, flood = 65.1%, liq load = 36.3 m³/(m²·h)
  ✓ kG·a_w = 2.585 mol/(m³·s·atm) — in expected range
    CO₂ removed (gas-phase):  1.691679e-04 mol/s
    CO₂ gained  (liquid):     1.691686e-04 mol/s
    CO₂ balance error:        4.1762e-06  ✓
    H₂O gained  (gas-phase):  3.460002e-03 mol/s
    H₂O ∫N_H2O dz:           3.460002e-03 mol/s
    H₂O balance error:        1.2111e-09  ✓

  ─── TEST 2: Limiting Cases ───
  Auto-size (gas flooding): D = 29 cm, u_G = 14.7 cm/s, flood = 65.1%, liq load = 36.3 m³/(m²·h)
  ✓ kG·a_w = 2.585 mol/(m³·s·atm) — in expected range
    Z = 0.1 m:  η = 3.78%  ✓ (should be low)
  Auto-size (gas flooding): D = 29 cm, u_G = 14.7 cm/s, flood = 65.1%, liq load = 36.3 m³/(m²·h)
  ✓ kG·a_w = 2.585 mol/(m³·s·atm) — in expected range
    Z = 30  m:  η = 100.00%  ✓ (should be ~100%)
  Auto-size (gas flooding): D = 29 cm, u_G = 14.7 cm/s, flood = 65.1